# 🌊 Machine Learning for Flood Prediction — Medjerda River Basin, Tunisia
## LSTM · Random Forest · XGBoost | Multi-Horizon EWS | 1970–2024

**Authors:** Cyrine Belhadj • Noomene Fehri • Mohamed Mchiri • Habib Ben Boubaker • Aissa Hlimi • Aya Nasrallah  
**Affiliation:** DRE, FLAHM, ENIT, Tunis, Tunisia

**Data source:** `Medjerda_Historical_Hydrological_Database.xlsx` (DRE — Direction des Ressources en Eau)  
**Reference station:** Medjez el-Bab | Period: 1970–2024 | 55 hydrological years (Sep–Aug)

---
**Installation (run once):**
```bash
pip install tensorflow scikit-learn xgboost shap matplotlib seaborn scipy pandas numpy openpyxl pymannkendall
```

## Cell 1 — Library Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import pandas as pd
from math import sqrt
from pathlib import Path

# Visualisation
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Patch
from matplotlib.lines import Line2D
import seaborn as sns

# ML — sklearn
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (mean_squared_error, mean_absolute_error,
                              r2_score, confusion_matrix, classification_report)
from sklearn.inspection import permutation_importance

# ML — XGBoost
from xgboost import XGBRegressor

# ML — TensorFlow/Keras
import tensorflow as tf
tf.random.set_seed(42)
np.random.seed(42)
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout, LSTM, BatchNormalization
from tensorflow.keras.callbacks import Callback, EarlyStopping, ReduceLROnPlateau

# SHAP
import shap

# Statistics
from scipy import stats
from scipy.stats import genextreme, gumbel_r
from scipy.optimize import curve_fit

# Mann-Kendall
try:
    import pymannkendall as mk
    MK_AVAILABLE = True
except ImportError:
    MK_AVAILABLE = False
    print('⚠️  pymannkendall not installed — MK test will use scipy fallback')

print('✅ All libraries loaded')
print(f'   TensorFlow : {tf.__version__}')
print(f'   NumPy      : {np.__version__}')
print(f'   Pandas     : {pd.__version__}')

## Cell 2 — Load Database

In [ ]:
# ── Path to the database (same folder as this notebook by default) ────────────
DB_PATH = Path('Medjerda_Historical_Hydrological_Database.xlsx')
assert DB_PATH.exists(), f'❌ Database not found at {DB_PATH.resolve()}. Place the xlsx next to this notebook.'

# ── Load all sheets (header on row 3, i.e. index 2) ─────────────────────────
df_flow   = pd.read_excel(DB_PATH, sheet_name='Monthly_Flow_Level',  header=2)
df_rain   = pd.read_excel(DB_PATH, sheet_name='Monthly_Rainfall',    header=2)
df_annual = pd.read_excel(DB_PATH, sheet_name='Annual_Statistics',   header=2)
df_flood  = pd.read_excel(DB_PATH, sheet_name='Flood_Events',        header=2)
df_ml     = pd.read_excel(DB_PATH, sheet_name='ML_Ready_Features',   header=2)

# ── Parse dates ───────────────────────────────────────────────────────────────
df_ml['Date'] = pd.to_datetime(df_ml['Date'])
df_ml = df_ml.sort_values('Date').reset_index(drop=True)

# ── Reference station: Medjez el-Bab ─────────────────────────────────────────
STATION = 'Medjez el-Bab'
df_flow_mjb   = df_flow[df_flow['Station'] == STATION].copy()
df_rain_mjb   = df_rain[df_rain['Station'] == STATION].copy()
df_annual_mjb = df_annual[df_annual['Station'] == STATION].copy()

MONTHS_ORDER = ['Sep','Oct','Nov','Dec','Jan','Feb','Mar','Apr','May','Jun','Jul','Aug']

print('✅ Database loaded successfully')
print(f'   Monthly_Flow_Level  : {df_flow.shape[0]:4d} rows | {df_flow["Station"].nunique()} stations')
print(f'   Monthly_Rainfall    : {df_rain.shape[0]:4d} rows | {df_rain["Station"].nunique()} stations')
print(f'   Annual_Statistics   : {df_annual.shape[0]:4d} rows')
print(f'   Flood_Events        : {df_flood.shape[0]:4d} events')
print(f'   ML_Ready_Features   : {df_ml.shape[0]:4d} rows | Period: {df_ml["Date"].min().date()} → {df_ml["Date"].max().date()}')
print(f'   Train/Val/Test split: {df_ml["Data_Split"].value_counts().to_dict()}')

## Cell 3 — Exploratory Data Analysis: Time Series & Statistics

In [ ]:
# ── Build tidy monthly time series for Medjez el-Bab ─────────────────────────
q_cols = ['Sep Q','Oct Q','Nov Q','Dec Q','Jan Q','Feb Q',
          'Mar Q','Apr Q','May Q','Jun Q','Jul Q','Aug Q']
p_cols = ['Sep','Oct','Nov','Dec','Jan','Feb','Mar','Apr','May','Jun','Jul','Aug']
month_map = dict(zip(q_cols, range(1, 13)))
MONTH_NUMS = [9,10,11,12,1,2,3,4,5,6,7,8]  # Sep=9 ... Aug=8

def melt_station(df_station, val_cols, month_nums, val_name):
    rows = []
    for _, r in df_station.iterrows():
        yr = int(str(r['Hydro Year']).split('-')[0])
        for ci, (col, mn) in enumerate(zip(val_cols, month_nums)):
            cal_yr = yr if mn >= 9 else yr + 1
            rows.append({'date': pd.Timestamp(cal_yr, mn, 1),
                         'hydro_year': r['Hydro Year'],
                         'month_num': mn,
                         val_name: r[col]})
    return pd.DataFrame(rows).sort_values('date').reset_index(drop=True)

ts_q = melt_station(df_flow_mjb,   q_cols, MONTH_NUMS, 'Q')
ts_p = melt_station(df_rain_mjb,   p_cols, MONTH_NUMS, 'P')
ts_q['P'] = ts_p['P'].values

print('── Descriptive Statistics: Monthly Flow Q (m³/s) — Medjez el-Bab ──')
print(ts_q['Q'].describe().round(2))
print(f'   Skewness : {ts_q["Q"].skew():.3f}   Kurtosis : {ts_q["Q"].kurtosis():.3f}')
print()
print('── Descriptive Statistics: Monthly Rainfall P (mm) — Medjez el-Bab ──')
print(ts_q['P'].describe().round(2))

# Annual maxima
ann_max_q = df_annual_mjb[['Hydro Year','Annual Max Q (m³/s)',
                            'Annual Total Rain (mm)',
                            'Exceeds Q2','Exceeds Q5','Exceeds Q10']].copy()
ann_max_q.columns = ['hydro_year','ann_max_Q','ann_rain','exc_Q2','exc_Q5','exc_Q10']
ann_max_q['year_int'] = ann_max_q['hydro_year'].str[:4].astype(int)

flood_years = ann_max_q[ann_max_q['exc_Q10'] == 'Yes']['year_int'].tolist()
print(f'\n── Flood years (Q > Q10): {flood_years}')

## Cell 4 — Figure 1: Basin Overview & Time Series

In [ ]:
fig = plt.figure(figsize=(18, 14))
gs0 = gridspec.GridSpec(3, 2, figure=fig, hspace=0.45, wspace=0.32)

# ── 1. Full monthly flow time series ─────────────────────────────────────────
ax1 = fig.add_subplot(gs0[0, :])
ax1.fill_between(ts_q['date'], ts_q['Q'], alpha=0.25, color='steelblue')
ax1.plot(ts_q['date'], ts_q['Q'], color='steelblue', lw=0.8)

Q2_thr, Q5_thr, Q10_thr = 340, 580, 780  # Medjez el-Bab thresholds
ax1.axhline(Q2_thr,  color='gold',   ls='--', lw=1.2, label=f'Q2  = {Q2_thr} m³/s')
ax1.axhline(Q5_thr,  color='orange', ls='--', lw=1.2, label=f'Q5  = {Q5_thr} m³/s')
ax1.axhline(Q10_thr, color='red',    ls='--', lw=1.2, label=f'Q10 = {Q10_thr} m³/s')
ax1.axvline(pd.Timestamp('1981-09-01'), color='gray', ls=':', lw=1.5,
            label='Sidi Salem Dam (1981)')

for fy in flood_years:
    ax1.axvspan(pd.Timestamp(fy, 10, 1), pd.Timestamp(fy+1, 4, 1),
                alpha=0.08, color='red')

ax1.set_title('Monthly Mean Streamflow — Medjez el-Bab (1970–2024)', fontsize=12, fontweight='bold')
ax1.set_ylabel('Discharge Q (m³/s)')
ax1.legend(fontsize=8, ncol=3)
ax1.grid(alpha=0.25)

# ── 2. Annual max flow trend ──────────────────────────────────────────────────
ax2 = fig.add_subplot(gs0[1, 0])
years_int = ann_max_q['year_int'].values
ann_max   = ann_max_q['ann_max_Q'].values
colors_bar = ['#C00000' if y in flood_years else '#4472C4' for y in years_int]
ax2.bar(years_int, ann_max, color=colors_bar, alpha=0.8, width=0.7)
# Trend line
z = np.polyfit(years_int, ann_max, 1)
p_line = np.poly1d(z)
ax2.plot(years_int, p_line(years_int), 'k--', lw=1.5, label=f'Trend: {z[0]:+.1f} m³/s/yr')
ax2.axhline(Q10_thr, color='red', ls=':', lw=1.2, label=f'Q10 = {Q10_thr}')
ax2.set_title('Annual Maximum Flow (1970–2024)', fontsize=11, fontweight='bold')
ax2.set_xlabel('Year')
ax2.set_ylabel('Q max (m³/s)')
ax2.legend(fontsize=8)
ax2.grid(alpha=0.25)

# ── 3. Annual rainfall trend ──────────────────────────────────────────────────
ax3 = fig.add_subplot(gs0[1, 1])
ann_rain = ann_max_q['ann_rain'].values
rain_mean = ann_rain.mean()
ax3.bar(years_int, ann_rain,
        color=['#2196F3' if v >= rain_mean else '#FF7043' for v in ann_rain],
        alpha=0.8, width=0.7)
zr = np.polyfit(years_int, ann_rain, 1)
pr = np.poly1d(zr)
ax3.plot(years_int, pr(years_int), 'k--', lw=1.5, label=f'Trend: {zr[0]:+.1f} mm/yr')
ax3.axhline(rain_mean, color='navy', ls=':', lw=1.2, label=f'Mean = {rain_mean:.0f} mm')
ax3.set_title('Annual Rainfall — Medjez el-Bab (1970–2024)', fontsize=11, fontweight='bold')
ax3.set_xlabel('Year')
ax3.set_ylabel('Annual Rainfall (mm)')
ax3.legend(fontsize=8)
ax3.grid(alpha=0.25)

# ── 4. Seasonal climatology ───────────────────────────────────────────────────
ax4 = fig.add_subplot(gs0[2, 0])
monthly_q_mean = ts_q.groupby('month_num')['Q'].agg(['mean','std'])
monthly_p_mean = ts_q.groupby('month_num')['P'].agg(['mean','std'])
idx_order = [9,10,11,12,1,2,3,4,5,6,7,8]
mq = monthly_q_mean.reindex(idx_order)
mp = monthly_p_mean.reindex(idx_order)
x = range(12)
ax4b = ax4.twinx()
ax4.bar(x, mp['mean'], color='steelblue', alpha=0.5, width=0.6, label='Rainfall P (mm)')
ax4b.plot(x, mq['mean'], 'r-o', lw=2, ms=5, label='Flow Q (m³/s)')
ax4b.fill_between(x, mq['mean']-mq['std'], mq['mean']+mq['std'], alpha=0.15, color='red')
ax4.set_xticks(x); ax4.set_xticklabels(MONTHS_ORDER, fontsize=8)
ax4.set_ylabel('Rainfall (mm)', color='steelblue')
ax4b.set_ylabel('Flow Q (m³/s)', color='red')
ax4.set_title('Mean Seasonal Cycle (1970–2024)', fontsize=11, fontweight='bold')
lines1, labels1 = ax4.get_legend_handles_labels()
lines2, labels2 = ax4b.get_legend_handles_labels()
ax4.legend(lines1+lines2, labels1+labels2, fontsize=8)
ax4.grid(alpha=0.2)

# ── 5. P–Q scatter with flood highlight ──────────────────────────────────────
ax5 = fig.add_subplot(gs0[2, 1])
flood_mask = ts_q['Q'] >= Q5_thr
ax5.scatter(ts_q.loc[~flood_mask,'P'], ts_q.loc[~flood_mask,'Q'],
            alpha=0.3, s=12, color='steelblue', label='Normal')
ax5.scatter(ts_q.loc[flood_mask,'P'], ts_q.loc[flood_mask,'Q'],
            alpha=0.8, s=40, color='red', marker='*', label=f'Flood (Q≥Q5={Q5_thr})')
ax5.set_xlabel('Monthly Rainfall P (mm)')
ax5.set_ylabel('Monthly Flow Q (m³/s)')
ax5.set_title('Rainfall–Runoff Scatter (Monthly)', fontsize=11, fontweight='bold')
ax5.legend(fontsize=8)
ax5.grid(alpha=0.25)

fig.suptitle('Medjerda Basin — Medjez el-Bab | Exploratory Data Analysis (1970–2024)',
             fontsize=14, fontweight='bold', y=1.01)
plt.savefig('Fig1_EDA_TimeSeries.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 1 saved → Fig1_EDA_TimeSeries.png')

## Cell 5 — Figure 2: Mann-Kendall Trend & Non-Stationarity Analysis

In [ ]:
# ── Mann-Kendall trend test (annual maxima & annual rainfall) ─────────────────
def mk_test_fallback(x):
    """Spearman correlation as MK proxy if pymannkendall unavailable."""
    t = np.arange(len(x))
    rho, p = stats.spearmanr(t, x)
    trend = 'increasing' if rho > 0 else 'decreasing'
    return type('obj', (object,), {'trend': trend, 'p': p, 'Tau': rho,
                                   'slope': np.polyfit(t, x, 1)[0]})()

def run_mk(series, name):
    if MK_AVAILABLE:
        res = mk.original_test(series)
        slope = mk.sens_slope(series).slope
    else:
        res = mk_test_fallback(series)
        slope = res.slope
    sig = '***' if res.p < 0.001 else ('**' if res.p < 0.01 else ('*' if res.p < 0.05 else 'n.s.'))
    print(f'  {name:<35} trend={res.trend:<12} τ={res.Tau:+.3f}  p={res.p:.4f} {sig}  slope={slope:+.2f}/yr')
    return res, slope

print('── Mann-Kendall Trend Tests ─────────────────────────────────────────────')
print(f'  (*** p<0.001  ** p<0.01  * p<0.05  n.s. not significant)')
mk_qmax,  sl_qmax  = run_mk(ann_max_q['ann_max_Q'].dropna().values,    'Annual Max Flow (m³/s)')
mk_rain,  sl_rain  = run_mk(ann_max_q['ann_rain'].dropna().values,      'Annual Rainfall (mm)')
mk_q_all, sl_q_all = run_mk(ts_q['Q'].values,                           'Monthly Flow (all)')

# ── Pettitt change-point test (simplified via cumsum) ─────────────────────────
def pettitt_breakpoint(x):
    n = len(x)
    U = np.array([np.sum(np.sign(x[t] - x[:t])) for t in range(1, n)])
    idx = np.argmax(np.abs(U))
    return idx + 1

bp_q = pettitt_breakpoint(ann_max_q['ann_max_Q'].dropna().values)
bp_yr = ann_max_q['year_int'].iloc[bp_q]
print(f'\n  Pettitt change-point (Annual Max Q): year ≈ {bp_yr}')

# ── Decadal box plots ─────────────────────────────────────────────────────────
ts_q['decade'] = ((ts_q['date'].dt.year // 10) * 10).astype(str) + 's'

fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Decadal flow distribution
ax = axes[0]
decade_order = sorted(ts_q['decade'].unique())
ts_q.boxplot(column='Q', by='decade', ax=ax, positions=range(len(decade_order)),
             notch=False, sym='+')
ax.set_xticklabels(decade_order, rotation=30)
ax.set_title('Decadal Flow Distribution — Medjez el-Bab', fontsize=10, fontweight='bold')
ax.set_ylabel('Monthly Q (m³/s)')
ax.grid(alpha=0.3)
plt.sca(ax); plt.title('')

# Annual max flow with breakpoint
ax = axes[1]
ax.plot(years_int, ann_max, 'b-o', ms=4, lw=1, label='Annual Max Q')
ax.axvline(bp_yr, color='red', ls='--', lw=1.8,
           label=f'Change-point ≈ {bp_yr}')
m1 = ann_max[:bp_q].mean()
m2 = ann_max[bp_q:].mean()
ax.axhline(m1, xmin=0, xmax=(bp_q/len(years_int)), color='steelblue', ls=':', lw=1.8,
           label=f'Mean pre: {m1:.0f}')
ax.axhline(m2, xmin=(bp_q/len(years_int)), xmax=1, color='orange', ls=':', lw=1.8,
           label=f'Mean post: {m2:.0f}')
ax.set_title('Annual Max Flow + Pettitt Break', fontsize=10, fontweight='bold')
ax.set_xlabel('Year'); ax.set_ylabel('Q max (m³/s)')
ax.legend(fontsize=8); ax.grid(alpha=0.25)

# ACF of monthly flow (manual)
ax = axes[2]
q_vals = ts_q['Q'].values
nlags = 24
acf = [np.corrcoef(q_vals[:-lag], q_vals[lag:])[0,1] for lag in range(1, nlags+1)]
ci = 1.96 / np.sqrt(len(q_vals))
ax.bar(range(1, nlags+1), acf, color=['steelblue' if abs(a) > ci else 'lightgray' for a in acf])
ax.axhline(ci,  color='red', ls='--', lw=1)
ax.axhline(-ci, color='red', ls='--', lw=1)
ax.axhline(0,   color='black', lw=0.8)
ax.set_title('ACF — Monthly Flow (lags 1–24 months)', fontsize=10, fontweight='bold')
ax.set_xlabel('Lag (months)'); ax.set_ylabel('Autocorrelation')
ax.grid(alpha=0.25)

fig.suptitle('Figure 2 — Trend Analysis & Non-Stationarity Tests', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('Fig2_TrendAnalysis.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 2 saved → Fig2_TrendAnalysis.png')

## Cell 6 — Return Period Calibration (GEV & Gumbel)

In [ ]:
# ── Fit GEV and Gumbel to annual maxima ───────────────────────────────────────
ann_max_clean = ann_max_q['ann_max_Q'].dropna().values

# GEV fit (L-moments equivalent via scipy MLE)
c_gev, loc_gev, scale_gev = genextreme.fit(ann_max_clean)
loc_gum, scale_gum = gumbel_r.fit(ann_max_clean)

return_periods = [2, 5, 10, 20, 50, 100]
exceedance_probs = [1 - 1/rp for rp in return_periods]

q_gev  = genextreme.ppf(exceedance_probs, c_gev,  loc=loc_gev,  scale=scale_gev)
q_gum  = gumbel_r.ppf(exceedance_probs, loc=loc_gum, scale=scale_gum)

# 95% CI via bootstrap
N_BOOT = 500
boot_gev = []
for _ in range(N_BOOT):
    sample = np.random.choice(ann_max_clean, size=len(ann_max_clean), replace=True)
    c_, l_, s_ = genextreme.fit(sample)
    boot_gev.append(genextreme.ppf(exceedance_probs, c_, loc=l_, scale=s_))
boot_gev = np.array(boot_gev)
ci_low_gev  = np.percentile(boot_gev, 2.5, axis=0)
ci_high_gev = np.percentile(boot_gev, 97.5, axis=0)

print('── Return Period Thresholds — Medjez el-Bab ────────────────────────────')
print(f'{"T (yr)":>8} {"GEV (m³/s)":>12} {"CI 2.5%":>10} {"CI 97.5%":>10} {"Gumbel":>10}')
print('-' * 55)
for i, rp in enumerate(return_periods):
    print(f'{rp:>8} {q_gev[i]:>12.1f} {ci_low_gev[i]:>10.1f} {ci_high_gev[i]:>10.1f} {q_gum[i]:>10.1f}')

# Store Q2/Q5/Q10 for EWS calibration
Q2_GEV  = float(q_gev[0])
Q5_GEV  = float(q_gev[1])
Q10_GEV = float(q_gev[2])
print(f'\n  EWS thresholds (GEV): Q2={Q2_GEV:.0f}  Q5={Q5_GEV:.0f}  Q10={Q10_GEV:.0f} m³/s')

# ── Plot return period curve ──────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
rp_range = np.logspace(np.log10(1.01), np.log10(200), 200)
ep_range = 1 - 1/rp_range
ax.semilogx(rp_range, genextreme.ppf(ep_range, c_gev, loc=loc_gev, scale=scale_gev),
            'b-', lw=2, label='GEV fit')
ax.semilogx(rp_range, gumbel_r.ppf(ep_range, loc=loc_gum, scale=scale_gum),
            'r--', lw=2, label='Gumbel fit')
ax.fill_between(rp_range,
                genextreme.ppf(ep_range, *genextreme.fit(ann_max_clean*(1-0.05))),
                genextreme.ppf(ep_range, *genextreme.fit(ann_max_clean*(1+0.05))),
                alpha=0.15, color='blue', label='95% CI (approx.)')
# Empirical Weibull positions
n = len(ann_max_clean)
empirical_rp = (n + 1) / (n + 1 - np.arange(1, n + 1))
ax.scatter(empirical_rp, np.sort(ann_max_clean), color='black', s=18, zorder=5, label='Empirical')
for rp, qg in zip([2,5,10], [Q2_GEV, Q5_GEV, Q10_GEV]):
    ax.axhline(qg, color='gray', ls=':', lw=1)
    ax.text(150, qg+5, f'Q{rp}={qg:.0f}', fontsize=8, color='gray')
ax.set_xlabel('Return Period T (years)')
ax.set_ylabel('Peak Discharge Q (m³/s)')
ax.set_title('Figure 3 — Return Period Curves — Medjez el-Bab', fontsize=12, fontweight='bold')
ax.legend(fontsize=9)
ax.grid(alpha=0.3, which='both')
plt.tight_layout()
plt.savefig('Fig3_ReturnPeriods.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 3 saved → Fig3_ReturnPeriods.png')

## Cell 7 — Feature Engineering & ML Preprocessing

In [ ]:
# ── Use the pre-built ML_Ready_Features sheet ────────────────────────────────
ml = df_ml.copy()

# Drop rows with NaN in essential columns (first few rows lack lag features)
FEAT_COLS = ['Q_t1 (m³/s)','Q_t2 (m³/s)','Q_t3 (m³/s)',
             'P_t (mm)','P_t1 (mm)','P_t2 (mm)','P_t3 (mm)',
             'API_5 (mm)','API_10 (mm)','API_30 (mm)',
             'H_t (m)','Month_Sin','Month_Cos','Is_Flood_Season']
TARGET = 'Q_t (m³/s)'

ml = ml.dropna(subset=FEAT_COLS + [TARGET]).reset_index(drop=True)

# Add Sidi Salem level (fill pre-1981 with 0)
ml['Sidi_Salem_Level (m)'] = ml['Sidi_Salem_Level (m)'].fillna(0)
FEAT_COLS_FULL = FEAT_COLS + ['Sidi_Salem_Level (m)']

# ── Chronological train/val/test split ───────────────────────────────────────
train_mask = ml['Data_Split'] == 'Train'
val_mask   = ml['Data_Split'] == 'Validation'
test_mask  = ml['Data_Split'] == 'Test'

X_train = ml.loc[train_mask, FEAT_COLS_FULL].values
y_train = ml.loc[train_mask, TARGET].values
X_val   = ml.loc[val_mask,   FEAT_COLS_FULL].values
y_val   = ml.loc[val_mask,   TARGET].values
X_test  = ml.loc[test_mask,  FEAT_COLS_FULL].values
y_test  = ml.loc[test_mask,  TARGET].values

dates_train = ml.loc[train_mask, 'Date'].values
dates_val   = ml.loc[val_mask,   'Date'].values
dates_test  = ml.loc[test_mask,  'Date'].values

# ── Scaling ──────────────────────────────────────────────────────────────────
scaler_X  = MinMaxScaler()
scaler_y  = MinMaxScaler()
zscore_X  = StandardScaler()

X_train_mm = scaler_X.fit_transform(X_train)
X_val_mm   = scaler_X.transform(X_val)
X_test_mm  = scaler_X.transform(X_test)

y_train_mm = scaler_y.fit_transform(y_train.reshape(-1,1)).flatten()
y_val_mm   = scaler_y.transform(y_val.reshape(-1,1)).flatten()
y_test_mm  = scaler_y.transform(y_test.reshape(-1,1)).flatten()

# Z-score for RF/XGBoost
X_train_z = zscore_X.fit_transform(X_train)
X_val_z   = zscore_X.transform(X_val)
X_test_z  = zscore_X.transform(X_test)

# LSTM: reshape to [samples, timesteps=1, features]
X_train_lstm = X_train_mm.reshape(X_train_mm.shape[0], 1, X_train_mm.shape[1])
X_val_lstm   = X_val_mm.reshape(X_val_mm.shape[0],   1, X_val_mm.shape[1])
X_test_lstm  = X_test_mm.reshape(X_test_mm.shape[0], 1, X_test_mm.shape[1])

N_FEATURES = len(FEAT_COLS_FULL)

print('✅ Feature matrix ready')
print(f'   Features       : {FEAT_COLS_FULL}')
print(f'   Train samples  : {X_train.shape[0]}  ({dates_train[0].astype("datetime64[M]"):} → {dates_train[-1].astype("datetime64[M]")})')
print(f'   Val samples    : {X_val.shape[0]}')
print(f'   Test samples   : {X_test.shape[0]}')
print(f'   Target (Q) stats — mean={y_train.mean():.1f}  std={y_train.std():.1f}  max={y_train.max():.1f} m³/s')

## Cell 8 — Figure 4: Feature Correlation Heatmap

In [ ]:
feat_df = pd.DataFrame(X_train, columns=FEAT_COLS_FULL)
feat_df[TARGET] = y_train

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Correlation heatmap
corr = feat_df.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, cmap='RdBu_r', center=0, ax=axes[0],
            annot=True, fmt='.2f', annot_kws={'size': 7},
            linewidths=0.4, cbar_kws={'shrink': 0.8})
axes[0].set_title('Feature Correlation Matrix', fontsize=11, fontweight='bold')
axes[0].tick_params(axis='x', rotation=45, labelsize=7)
axes[0].tick_params(axis='y', labelsize=7)

# Pearson correlation with target
corr_target = feat_df.corr()[TARGET].drop(TARGET).sort_values()
colors_bar = ['#C00000' if c > 0 else '#4472C4' for c in corr_target]
axes[1].barh(range(len(corr_target)), corr_target.values, color=colors_bar, alpha=0.85)
axes[1].set_yticks(range(len(corr_target)))
axes[1].set_yticklabels(corr_target.index, fontsize=8)
axes[1].axvline(0, color='black', lw=0.8)
axes[1].set_xlabel('Pearson r with Q_t')
axes[1].set_title(f'Feature Correlation with Target ({TARGET})', fontsize=11, fontweight='bold')
axes[1].grid(alpha=0.3, axis='x')

fig.suptitle('Figure 4 — Feature Analysis', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('Fig4_FeatureCorrelation.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 4 saved')

## Cell 9 — RMSE Callback & Metrics Utilities

In [ ]:
class RMSEHistory(Callback):
    def __init__(self, tX, ty, vX, vy, scaler_y):
        super().__init__()
        self.tX, self.ty = tX, ty
        self.vX, self.vy = vX, vy
        self.scaler_y = scaler_y
        self.train_rmse, self.val_rmse = [], []

    def on_epoch_end(self, epoch, logs=None):
        tp = self.scaler_y.inverse_transform(
                self.model.predict(self.tX, verbose=0)).flatten()
        vp = self.scaler_y.inverse_transform(
                self.model.predict(self.vX, verbose=0)).flatten()
        ty_ = self.scaler_y.inverse_transform(self.ty.reshape(-1,1)).flatten()
        vy_ = self.scaler_y.inverse_transform(self.vy.reshape(-1,1)).flatten()
        self.train_rmse.append(sqrt(mean_squared_error(ty_, tp)))
        self.val_rmse.append(  sqrt(mean_squared_error(vy_, vp)))


def nse(obs, sim):
    """Nash-Sutcliffe Efficiency."""
    return 1 - np.sum((obs - sim)**2) / np.sum((obs - np.mean(obs))**2)

def kge(obs, sim):
    """Kling-Gupta Efficiency."""
    r  = np.corrcoef(obs, sim)[0,1]
    a  = np.std(sim) / np.std(obs)
    b  = np.mean(sim) / np.mean(obs)
    return 1 - np.sqrt((r-1)**2 + (a-1)**2 + (b-1)**2)

def pbias(obs, sim):
    """Percent bias."""
    return 100 * np.sum(obs - sim) / np.sum(obs)

def flood_metrics(obs, sim, threshold):
    """POD, FAR, CSI for threshold exceedance."""
    obs_b = (obs >= threshold).astype(int)
    sim_b = (sim >= threshold).astype(int)
    TP = np.sum((obs_b == 1) & (sim_b == 1))
    FP = np.sum((obs_b == 0) & (sim_b == 1))
    FN = np.sum((obs_b == 1) & (sim_b == 0))
    POD = TP / (TP + FN + 1e-9)
    FAR = FP / (TP + FP + 1e-9)
    CSI = TP / (TP + FP + FN + 1e-9)
    return POD, FAR, CSI

def print_metrics(obs, pred, label, thr_dict=None):
    rmse_ = sqrt(mean_squared_error(obs, pred))
    mae_  = mean_absolute_error(obs, pred)
    r2_   = r2_score(obs, pred)
    nse_  = nse(obs, pred)
    kge_  = kge(obs, pred)
    pb_   = pbias(obs, pred)
    print(f'  {label:<28} NSE={nse_:.3f}  KGE={kge_:.3f}  RMSE={rmse_:7.2f}  MAE={mae_:6.2f}  PBIAS={pb_:+.1f}%  R²={r2_:.3f}')
    if thr_dict:
        for nm, thr in thr_dict.items():
            pod, far, csi = flood_metrics(obs, pred, thr)
            print(f'  {'':28} [{nm}] POD={pod:.2f}  FAR={far:.2f}  CSI={csi:.2f}')
    return {'NSE': nse_, 'KGE': kge_, 'RMSE': rmse_, 'MAE': mae_, 'PBIAS': pb_, 'R2': r2_}

THRESHOLDS = {'Q2': Q2_GEV, 'Q5': Q5_GEV, 'Q10': Q10_GEV}
print('✅ Metrics utilities ready')

## Cell 10 — Model 1: LSTM (2-layer stacked)

In [ ]:
def build_lstm(n_features):
    model = Sequential([
        LSTM(128, input_shape=(1, n_features), return_sequences=True),
        Dropout(0.2),
        LSTM(64),
        Dropout(0.2),
        BatchNormalization(),
        Dense(32, activation='relu'),
        Dense(1),
    ])
    model.compile(
        loss='mse',
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3)
    )
    return model

print('Training LSTM — 200 epochs (with early stopping) …')

lstm_model = build_lstm(N_FEATURES)
rmse_cb    = RMSEHistory(X_train_lstm, y_train_mm, X_val_lstm, y_val_mm, scaler_y)
es_cb      = EarlyStopping(monitor='val_loss', patience=25, restore_best_weights=True)
lr_cb      = ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=10, min_lr=1e-5)

history = lstm_model.fit(
    X_train_lstm, y_train_mm,
    epochs=200, batch_size=16,
    validation_data=(X_val_lstm, y_val_mm),
    callbacks=[rmse_cb, es_cb, lr_cb],
    verbose=1, shuffle=False
)

# Inverse-scale predictions
def inv_y(arr):
    return scaler_y.inverse_transform(np.array(arr).reshape(-1,1)).flatten()

lstm_train_pred = inv_y(lstm_model.predict(X_train_lstm, verbose=0))
lstm_val_pred   = inv_y(lstm_model.predict(X_val_lstm,   verbose=0))
lstm_test_pred  = inv_y(lstm_model.predict(X_test_lstm,  verbose=0))

print('\n✅ LSTM training complete')
print('── LSTM Performance ────────────────────────────────────────────────────')
lstm_metrics_train = print_metrics(y_train, lstm_train_pred, 'LSTM (train)')
lstm_metrics_val   = print_metrics(y_val,   lstm_val_pred,   'LSTM (val)')
lstm_metrics_test  = print_metrics(y_test,  lstm_test_pred,  'LSTM (test)', THRESHOLDS)

## Cell 11 — Model 2: Random Forest

In [ ]:
print('Training Random Forest …')

rf_model = RandomForestRegressor(
    n_estimators=400,
    max_depth=None,
    min_samples_split=5,
    min_samples_leaf=2,
    max_features='sqrt',
    n_jobs=-1,
    random_state=42
)
# Train on train+val combined (standard for RF)
X_trainval_z = np.vstack([X_train_z, X_val_z])
y_trainval   = np.concatenate([y_train, y_val])
rf_model.fit(X_trainval_z, y_trainval)

rf_train_pred = rf_model.predict(X_train_z)
rf_val_pred   = rf_model.predict(X_val_z)
rf_test_pred  = rf_model.predict(X_test_z)

print('✅ Random Forest training complete')
print('── RF Performance ──────────────────────────────────────────────────────')
rf_metrics_train = print_metrics(y_train, rf_train_pred, 'RF (train)')
rf_metrics_val   = print_metrics(y_val,   rf_val_pred,   'RF (val)')
rf_metrics_test  = print_metrics(y_test,  rf_test_pred,  'RF (test)', THRESHOLDS)

# MDI feature importance
rf_importance = pd.Series(
    rf_model.feature_importances_, index=FEAT_COLS_FULL
).sort_values(ascending=False)
print('\n── RF Feature Importance (MDI) ─────────────────────────────────────────')
print(rf_importance.round(4).to_string())

## Cell 12 — Model 3: XGBoost

In [ ]:
print('Training XGBoost …')

xgb_model = XGBRegressor(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=6,
    min_child_weight=3,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    random_state=42,
    eval_metric='rmse',
    early_stopping_rounds=30,
    verbosity=0,
    n_jobs=-1
)
xgb_model.fit(
    X_train_z, y_train,
    eval_set=[(X_val_z, y_val)],
    verbose=False
)

xgb_train_pred = xgb_model.predict(X_train_z)
xgb_val_pred   = xgb_model.predict(X_val_z)
xgb_test_pred  = xgb_model.predict(X_test_z)

print('✅ XGBoost training complete')
print('── XGBoost Performance ─────────────────────────────────────────────────')
xgb_metrics_train = print_metrics(y_train, xgb_train_pred, 'XGBoost (train)')
xgb_metrics_val   = print_metrics(y_val,   xgb_val_pred,   'XGBoost (val)')
xgb_metrics_test  = print_metrics(y_test,  xgb_test_pred,  'XGBoost (test)', THRESHOLDS)

## Cell 13 — Figure 5: Comparative Metrics Table & Bar Chart

In [ ]:
metrics_df = pd.DataFrame({
    'Model': ['LSTM', 'Random Forest', 'XGBoost'],
    'NSE':   [lstm_metrics_test['NSE'],  rf_metrics_test['NSE'],  xgb_metrics_test['NSE']],
    'KGE':   [lstm_metrics_test['KGE'],  rf_metrics_test['KGE'],  xgb_metrics_test['KGE']],
    'RMSE':  [lstm_metrics_test['RMSE'], rf_metrics_test['RMSE'], xgb_metrics_test['RMSE']],
    'MAE':   [lstm_metrics_test['MAE'],  rf_metrics_test['MAE'],  xgb_metrics_test['MAE']],
    'PBIAS': [lstm_metrics_test['PBIAS'],rf_metrics_test['PBIAS'],xgb_metrics_test['PBIAS']],
    'R2':    [lstm_metrics_test['R2'],   rf_metrics_test['R2'],   xgb_metrics_test['R2']],
}).set_index('Model')

print('─' * 80)
print('TABLE 3 — Comparative Model Performance on Test Set (Medjez el-Bab)')
print('─' * 80)
print(metrics_df.round(3).to_string())
print('─' * 80)

fig, axes = plt.subplots(2, 3, figsize=(16, 8))
axes = axes.flatten()
COLORS = ['#2196F3', '#4CAF50', '#FF5722']
metric_names = ['NSE', 'KGE', 'RMSE (m³/s)', 'MAE (m³/s)', 'PBIAS (%)', 'R²']
metric_keys  = ['NSE', 'KGE', 'RMSE', 'MAE', 'PBIAS', 'R2']

for i, (mk_, label) in enumerate(zip(metric_keys, metric_names)):
    vals = metrics_df[mk_].values
    bars = axes[i].bar(metrics_df.index, vals, color=COLORS, alpha=0.85, width=0.5)
    axes[i].set_title(label, fontsize=11, fontweight='bold')
    axes[i].set_ylabel(label)
    axes[i].grid(alpha=0.3, axis='y')
    for bar, v in zip(bars, vals):
        axes[i].text(bar.get_x() + bar.get_width()/2,
                     bar.get_height() + abs(bar.get_height())*0.02,
                     f'{v:.3f}', ha='center', va='bottom', fontsize=9)
    if mk_ in ['NSE','KGE','R2']:
        axes[i].axhline(0.75, color='gray', ls=':', lw=1.2)

fig.suptitle('Figure 5 — Model Comparison on Test Set', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('Fig5_ModelComparison.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 5 saved')

## Cell 14 — Figure 6: Hydrographs — Observed vs Predicted

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(16, 14), sharex=False)

plot_sets = [
    ('Train', dates_train, y_train, lstm_train_pred, rf_train_pred, xgb_train_pred),
    ('Validation', dates_val,   y_val,   lstm_val_pred,   rf_val_pred,   xgb_val_pred),
    ('Test',  dates_test,  y_test,  lstm_test_pred,  rf_test_pred,  xgb_test_pred),
]

for ax, (split, dates, obs, lstm_p, rf_p, xgb_p) in zip(axes, plot_sets):
    dt = pd.to_datetime(dates)
    ax.fill_between(dt, 0, obs, alpha=0.2, color='black', label='Observed')
    ax.plot(dt, obs,    'k-',  lw=1.5, label='Observed',       alpha=0.9)
    ax.plot(dt, lstm_p, 'b--', lw=1.5, label='LSTM',           alpha=0.85)
    ax.plot(dt, rf_p,   'g-',  lw=1.5, label='Random Forest',  alpha=0.85)
    ax.plot(dt, xgb_p,  'r-',  lw=1.2, label='XGBoost',        alpha=0.75)
    ax.axhline(Q2_GEV,  color='gold',   ls=':', lw=1.2)
    ax.axhline(Q5_GEV,  color='orange', ls=':', lw=1.2)
    ax.axhline(Q10_GEV, color='red',    ls=':', lw=1.2)
    ax.text(dt[-1], Q2_GEV+5,  'Q2',  fontsize=7, color='goldenrod')
    ax.text(dt[-1], Q5_GEV+5,  'Q5',  fontsize=7, color='orange')
    ax.text(dt[-1], Q10_GEV+5, 'Q10', fontsize=7, color='red')
    ax.set_title(f'{split} Set — Observed vs Predicted', fontsize=11, fontweight='bold')
    ax.set_ylabel('Discharge Q (m³/s)')
    ax.legend(fontsize=8, ncol=4)
    ax.grid(alpha=0.25)

fig.suptitle('Figure 6 — Hydrographs: Observed vs LSTM / RF / XGBoost — Medjez el-Bab',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('Fig6_Hydrographs.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 6 saved')

## Cell 15 — Figure 7: SHAP Analysis (XGBoost)

In [ ]:
print('Computing SHAP values (XGBoost) …')
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test_z)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# SHAP summary bar
plt.sca(axes[0])
shap.summary_plot(shap_values, X_test_z, feature_names=FEAT_COLS_FULL,
                  plot_type='bar', show=False, color='steelblue')
axes[0].set_title('SHAP — Mean |SHAP| (XGBoost)', fontsize=11, fontweight='bold')

# SHAP beeswarm
plt.sca(axes[1])
shap.summary_plot(shap_values, X_test_z, feature_names=FEAT_COLS_FULL, show=False)
axes[1].set_title('SHAP — Beeswarm (XGBoost)', fontsize=11, fontweight='bold')

fig.suptitle('Figure 7 — SHAP Feature Importance — XGBoost', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('Fig7_SHAP.png', dpi=150, bbox_inches='tight')
plt.show()

shap_mean = pd.Series(
    np.abs(shap_values).mean(axis=0), index=FEAT_COLS_FULL
).sort_values(ascending=False)
print('\n── SHAP Importance Ranking ─────────────────────────────────────────────')
print(shap_mean.round(4).to_string())
print('\n✅ Figure 7 saved')

## Cell 16 — Multi-Horizon Forecast Performance (1h→24h proxy)

In [ ]:
# ── Simulate multi-horizon via rolling lag shift on test set ─────────────────
# At monthly resolution we proxy h=1,2,3,6,12 months ahead

HORIZONS = [1, 2, 3, 6, 12]  # months ahead
horizon_results = []

q_col_idx   = FEAT_COLS_FULL.index('Q_t1 (m³/s)')  # position of Q_t-1
p_col_idx   = FEAT_COLS_FULL.index('P_t (mm)')
api5_idx    = FEAT_COLS_FULL.index('API_5 (mm)')

full_q = ml['Q_t (m³/s)'].values
full_p = ml['P_t (mm)'].values

for h in HORIZONS:
    # Shift target h steps into the future
    y_shifted = np.roll(full_q, -h)
    valid_mask = np.zeros(len(full_q), dtype=bool)
    split_indices = ml[ml['Data_Split'] == 'Test'].index
    for idx in split_indices:
        if idx + h < len(full_q):
            valid_mask[idx] = True

    Xh = ml.loc[valid_mask, FEAT_COLS_FULL].values
    yh = y_shifted[valid_mask]

    Xh_z    = zscore_X.transform(Xh)
    Xh_mm   = scaler_X.transform(Xh)
    Xh_lstm = Xh_mm.reshape(Xh_mm.shape[0], 1, Xh_mm.shape[1])

    pred_lstm = inv_y(lstm_model.predict(Xh_lstm, verbose=0))
    pred_rf   = rf_model.predict(Xh_z)
    pred_xgb  = xgb_model.predict(Xh_z)

    for model_name, pred in [('LSTM', pred_lstm), ('RF', pred_rf), ('XGBoost', pred_xgb)]:
        nse_ = nse(yh, pred)
        kge_ = kge(yh, pred)
        rmse_= sqrt(mean_squared_error(yh, pred))
        horizon_results.append({'Horizon': h, 'Model': model_name,
                                  'NSE': nse_, 'KGE': kge_, 'RMSE': rmse_})

horizon_df = pd.DataFrame(horizon_results)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
MCOLORS = {'LSTM': '#2196F3', 'RF': '#4CAF50', 'XGBoost': '#FF5722'}
MARKERS = {'LSTM': 'o-', 'RF': 's--', 'XGBoost': '^:'}

for ax, metric in zip(axes, ['NSE', 'KGE', 'RMSE']):
    for model_name, group in horizon_df.groupby('Model'):
        ax.plot(group['Horizon'], group[metric],
                MARKERS[model_name], lw=2, ms=7,
                color=MCOLORS[model_name], label=model_name)
    if metric in ['NSE', 'KGE']:
        ax.axhline(0.75, color='gray', ls=':', lw=1.2, label='Target=0.75')
    ax.set_xlabel('Forecast Horizon (months)')
    ax.set_ylabel(metric)
    ax.set_title(f'{metric} vs Forecast Horizon', fontsize=11, fontweight='bold')
    ax.set_xticks(HORIZONS)
    ax.legend(fontsize=8)
    ax.grid(alpha=0.3)

fig.suptitle('Figure 8 — Multi-Horizon Performance: LSTM / RF / XGBoost',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('Fig8_MultiHorizon.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 8 saved')
print(horizon_df.pivot(index='Horizon', columns='Model', values='NSE').round(3).to_string())

## Cell 17 — Figure 9: Training Dynamics & Learning Curves

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history.history['loss'],     color='steelblue', lw=1.8, label='Train Loss (MSE)')
axes[0].plot(history.history['val_loss'], color='tomato',    lw=1.8, label='Val Loss (MSE)')
axes[0].set_title('LSTM — Training & Validation Loss', fontsize=11, fontweight='bold')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('MSE')
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

axes[1].plot(rmse_cb.train_rmse, color='steelblue', lw=1.8, label='Train RMSE (m³/s)')
axes[1].plot(rmse_cb.val_rmse,   color='tomato',    lw=1.8, label='Val RMSE (m³/s)')
axes[1].set_title('LSTM — RMSE per Epoch', fontsize=11, fontweight='bold')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('RMSE (m³/s)')
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

fig.suptitle('Figure 9 — LSTM Learning Curves', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('Fig9_LearningCurves.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 9 saved')

## Cell 18 — Early Warning System (4-Level EWS)

In [ ]:
# ── EWS: ensemble forecast + threshold classification ────────────────────────

# Ensemble: NSE-weighted average of 3 models on test set
nse_lstm = lstm_metrics_test['NSE']
nse_rf   = rf_metrics_test['NSE']
nse_xgb  = xgb_metrics_test['NSE']
total_nse = nse_lstm + nse_rf + nse_xgb
w_lstm, w_rf, w_xgb = nse_lstm/total_nse, nse_rf/total_nse, nse_xgb/total_nse

ensemble_pred = w_lstm*lstm_test_pred + w_rf*rf_test_pred + w_xgb*xgb_test_pred

print(f'Ensemble weights — LSTM: {w_lstm:.3f}  RF: {w_rf:.3f}  XGBoost: {w_xgb:.3f}')
ens_metrics = print_metrics(y_test, ensemble_pred, 'Ensemble (test)', THRESHOLDS)

# ── Alert level classification ────────────────────────────────────────────────
def assign_alert(q, q2=Q2_GEV, q5=Q5_GEV, q10=Q10_GEV):
    if   q >= q10: return 4, 'ROUGE',  'Urgence — Évacuation immédiate'
    elif q >= q5:  return 3, 'ORANGE', 'Alerte — Préparer évacuation'
    elif q >= q2:  return 2, 'JAUNE',  'Vigilance — Activation cellule DRE'
    else:          return 1, 'VERT',   'Surveillance standard'

test_df = pd.DataFrame({
    'Date':           pd.to_datetime(dates_test),
    'Q_obs (m³/s)':   y_test,
    'Q_LSTM':         lstm_test_pred,
    'Q_RF':           rf_test_pred,
    'Q_XGBoost':      xgb_test_pred,
    'Q_Ensemble':     ensemble_pred,
})

for col in ['Q_obs (m³/s)', 'Q_Ensemble']:
    suffix = 'obs' if 'obs' in col else 'pred'
    test_df[[f'Level_{suffix}', f'Alert_{suffix}', f'Action_{suffix}']] = pd.DataFrame(
        test_df[col].apply(assign_alert).tolist(), index=test_df.index
    )

# Confusion between observed and predicted alert levels
print('\n── EWS Alert Level Confusion Matrix ────────────────────────────────────')
cm = confusion_matrix(test_df['Level_obs'], test_df['Level_pred'])
level_names = ['VERT','JAUNE','ORANGE','ROUGE']
# Keep only levels that appear
levels_present = sorted(test_df['Level_obs'].unique())
cm_df = pd.DataFrame(cm, 
    index=[level_names[i-1] for i in levels_present],
    columns=[level_names[i-1] for i in levels_present])
print(cm_df.to_string())

# ── EWS visualization ─────────────────────────────────────────────────────────
EWS_COLORS = {1: '#00B050', 2: '#FFFF00', 3: '#FFC000', 4: '#C00000'}

fig, axes = plt.subplots(3, 1, figsize=(16, 12))

# Panel 1: observed + ensemble + thresholds
ax = axes[0]
ax.plot(test_df['Date'], test_df['Q_obs (m³/s)'], 'k-', lw=2, label='Observed Q')
ax.plot(test_df['Date'], test_df['Q_Ensemble'],   'b--',lw=1.8, label='Ensemble Forecast')
ax.axhline(Q2_GEV,  color='gold',   ls='--', lw=1.2, label=f'Q2={Q2_GEV:.0f}')
ax.axhline(Q5_GEV,  color='orange', ls='--', lw=1.2, label=f'Q5={Q5_GEV:.0f}')
ax.axhline(Q10_GEV, color='red',    ls='--', lw=1.5, label=f'Q10={Q10_GEV:.0f}')
ax.fill_between(test_df['Date'], Q2_GEV,  Q5_GEV,  alpha=0.08, color='yellow')
ax.fill_between(test_df['Date'], Q5_GEV,  Q10_GEV, alpha=0.08, color='orange')
ax.fill_between(test_df['Date'], Q10_GEV, test_df['Q_obs (m³/s)'].max()*1.1, alpha=0.08, color='red')
ax.set_title('EWS — Test Set: Observed vs Ensemble Forecast with Alert Thresholds',
             fontsize=11, fontweight='bold')
ax.set_ylabel('Q (m³/s)'); ax.legend(fontsize=8, ncol=3); ax.grid(alpha=0.25)

# Panel 2: alert level strips (predicted)
ax2 = axes[1]
for i, row in test_df.iterrows():
    ax2.bar(row['Date'], row['Level_pred'], width=25,
            color=EWS_COLORS[row['Level_pred']], alpha=0.85, edgecolor='none')
ax2.set_title('EWS — Predicted Alert Levels (Ensemble)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Alert Level (1=Vert … 4=Rouge)')
ax2.set_yticks([1,2,3,4])
ax2.set_yticklabels(['VERT','JAUNE','ORANGE','ROUGE'])
legend_els = [Patch(color=EWS_COLORS[i], label=nm)
              for i, nm in enumerate(['VERT','JAUNE','ORANGE','ROUGE'], 1)]
ax2.legend(handles=legend_els, fontsize=8, ncol=4)
ax2.grid(alpha=0.2, axis='x')

# Panel 3: alert level comparison obs vs pred
ax3 = axes[2]
ax3.step(test_df['Date'], test_df['Level_obs'],  'k-',  lw=2, where='post', label='Observed Level')
ax3.step(test_df['Date'], test_df['Level_pred'], 'b--', lw=1.8, where='post', label='Predicted Level')
ax3.set_title('EWS — Alert Level: Observed vs Predicted', fontsize=11, fontweight='bold')
ax3.set_ylabel('Alert Level'); ax3.set_yticks([1,2,3,4])
ax3.set_yticklabels(['VERT','JAUNE','ORANGE','ROUGE'])
ax3.legend(fontsize=9); ax3.grid(alpha=0.25)

fig.suptitle('Figure 10 — Early Warning System (4-Level EWS) — Medjez el-Bab',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('Fig10_EWS.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 10 saved')

## Cell 19 — 10-Year Forecast (Monte Carlo, LSTM)

In [ ]:
FORECAST_MONTHS = 120
N_SIMULATIONS   = 50
N_IN            = X_train_mm.shape[1]  # = N_FEATURES

# ── Deterministic forecast ────────────────────────────────────────────────────
last_seq = X_test_mm[-1].copy()  # last observed feature vector
future_scaled = []

for _ in range(FORECAST_MONTHS):
    inp = last_seq.reshape(1, 1, N_IN)
    p   = lstm_model.predict(inp, verbose=0)[0][0]
    future_scaled.append(float(p))
    # Shift lag features: Q_t → Q_t1
    last_seq[0] = p   # update Q_t1 position

future_q = inv_y(np.array(future_scaled))
last_date = pd.to_datetime(dates_test[-1])
future_dates = pd.date_range(last_date + pd.DateOffset(months=1),
                              periods=FORECAST_MONTHS, freq='MS')

# ── Monte Carlo uncertainty ───────────────────────────────────────────────────
print('Running Monte Carlo simulations …')
all_sims = []
for sim_i in range(N_SIMULATIONS):
    seq = X_test_mm[-1].copy()
    sim = []
    for _ in range(FORECAST_MONTHS):
        inp = seq.reshape(1, 1, N_IN)
        p   = lstm_model(inp, training=True).numpy()[0][0]
        sim.append(float(np.clip(p, 0, 1)))
        seq[0] = p
    all_sims.append(inv_y(np.array(sim)))
    if (sim_i + 1) % 10 == 0:
        print(f'  {sim_i+1}/{N_SIMULATIONS} done …')

sims_arr = np.array(all_sims)
ci_low   = np.percentile(sims_arr,  5, axis=0)
ci_high  = np.percentile(sims_arr, 95, axis=0)

# ── Build output DataFrames ───────────────────────────────────────────────────
forecast_df = pd.DataFrame({
    'Date':           future_dates,
    'Forecast Q (m³/s)': np.round(future_q, 2),
    'CI_5 (m³/s)':    np.round(ci_low,   2),
    'CI_95 (m³/s)':   np.round(ci_high,  2),
})
forecast_df['Alert_Level'] = forecast_df['Forecast Q (m³/s)'].apply(
    lambda q: assign_alert(q)[1])

annual_fc = (forecast_df.groupby(forecast_df['Date'].dt.year)['Forecast Q (m³/s)']
             .agg(['mean','max']).reset_index())
annual_fc.columns = ['Year','Annual Mean Q (m³/s)','Annual Max Q (m³/s)']

print('\n✅ Forecast complete')
print(forecast_df[['Date','Forecast Q (m³/s)','CI_5 (m³/s)','CI_95 (m³/s)','Alert_Level']].head(24).to_string(index=False))

## Cell 20 — Figure 11: 10-Year Forecast Visualization

In [ ]:
fig = plt.figure(figsize=(18, 16))
gs0 = gridspec.GridSpec(3, 2, figure=fig, hspace=0.42, wspace=0.32)

# 1. Full series + forecast
ax1 = fig.add_subplot(gs0[0, :])
ax1.fill_between(ts_q['date'], ts_q['Q'], alpha=0.2, color='black')
ax1.plot(ts_q['date'], ts_q['Q'], 'k-', lw=0.9, label='Observed (1970–2024)')
ax1.fill_between(forecast_df['Date'], ci_low, ci_high, alpha=0.2, color='purple', label='90% CI')
ax1.plot(forecast_df['Date'], future_q, color='purple', lw=2, label='LSTM Forecast (2024–2034)')
ax1.axvline(pd.Timestamp('2024-09-01'), color='gray', ls=':', lw=1.5)
ax1.axhline(Q2_GEV,  color='gold',   ls='--', lw=1.2)
ax1.axhline(Q5_GEV,  color='orange', ls='--', lw=1.2)
ax1.axhline(Q10_GEV, color='red',    ls='--', lw=1.2)
ax1.set_title('Monthly Flow — Observed (1970–2024) + 10-Year LSTM Forecast',
              fontsize=12, fontweight='bold')
ax1.set_ylabel('Q (m³/s)'); ax1.legend(fontsize=9, ncol=3); ax1.grid(alpha=0.25)

# 2. Annual max forecast
ax2 = fig.add_subplot(gs0[1, 0])
bar_colors = ['#C00000' if v >= Q5_GEV else '#4472C4' for v in annual_fc['Annual Max Q (m³/s)']]
ax2.bar(annual_fc['Year'], annual_fc['Annual Max Q (m³/s)'], color=bar_colors, alpha=0.85, width=0.6)
ax2.axhline(Q5_GEV,  color='orange', ls='--', lw=1.2, label=f'Q5={Q5_GEV:.0f}')
ax2.axhline(Q10_GEV, color='red',    ls='--', lw=1.2, label=f'Q10={Q10_GEV:.0f}')
ax2.set_title('Forecasted Annual Max Flow (2025–2034)', fontsize=11, fontweight='bold')
ax2.set_ylabel('Q max (m³/s)'); ax2.legend(fontsize=8); ax2.grid(alpha=0.25)

# 3. Alert level distribution in forecast
ax3 = fig.add_subplot(gs0[1, 1])
alert_counts = forecast_df['Alert_Level'].value_counts()
level_order = ['VERT','JAUNE','ORANGE','ROUGE']
alert_counts = alert_counts.reindex([l for l in level_order if l in alert_counts.index])
alert_colors = {'VERT': '#00B050', 'JAUNE': '#FFFF00', 'ORANGE': '#FFC000', 'ROUGE': '#C00000'}
ax3.bar(alert_counts.index, alert_counts.values,
        color=[alert_colors[k] for k in alert_counts.index], alpha=0.85, width=0.5)
for i, (l, v) in enumerate(alert_counts.items()):
    ax3.text(i, v+0.5, f'{v}\n({v/len(forecast_df)*100:.0f}%)', ha='center', fontsize=9)
ax3.set_title('EWS Alert Level Distribution\n(120-month forecast)', fontsize=11, fontweight='bold')
ax3.set_ylabel('Number of months'); ax3.grid(alpha=0.25, axis='y')

# 4. Seasonal heatmap of forecast
ax4 = fig.add_subplot(gs0[2, :])
fc_heat = forecast_df.copy()
fc_heat['Year']  = fc_heat['Date'].dt.year
fc_heat['Month'] = fc_heat['Date'].dt.month
pivot = fc_heat.pivot_table(values='Forecast Q (m³/s)', index='Year', columns='Month')
pivot.columns = [pd.Timestamp(2000, m, 1).strftime('%b') for m in pivot.columns]
im = ax4.imshow(pivot.values, aspect='auto', cmap='YlOrRd')
ax4.set_xticks(range(pivot.shape[1]))
ax4.set_xticklabels(pivot.columns, fontsize=9)
ax4.set_yticks(range(len(pivot)))
ax4.set_yticklabels(pivot.index, fontsize=9)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        v = pivot.values[i, j]
        ax4.text(j, i, f'{v:.0f}', ha='center', va='center',
                 fontsize=7, color='white' if v > Q5_GEV else 'black')
ax4.set_title('Seasonal Heatmap — Forecasted Q (m³/s) 2025–2034', fontsize=11, fontweight='bold')
plt.colorbar(im, ax=ax4, label='Q (m³/s)', shrink=0.6)

fig.suptitle('Figure 11 — 10-Year Flow Forecast (LSTM) + EWS Alert Levels — Medjez el-Bab',
             fontsize=13, fontweight='bold')
plt.savefig('Fig11_10YearForecast.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Figure 11 saved')

## Cell 21 — Export All Results to Excel

In [ ]:
out_xlsx = 'ML_Results_Medjerda_FloodPrediction.xlsx'

with pd.ExcelWriter(out_xlsx, engine='openpyxl') as writer:

    # Sheet 1: Model comparison metrics
    metrics_df.round(4).to_excel(writer, sheet_name='Model_Metrics_Test')

    # Sheet 2: Multi-horizon
    horizon_df.pivot(index='Horizon', columns='Model', values='NSE').round(4).to_excel(
        writer, sheet_name='MultiHorizon_NSE')
    horizon_df.pivot(index='Horizon', columns='Model', values='KGE').round(4).to_excel(
        writer, sheet_name='MultiHorizon_KGE')

    # Sheet 3: Return period thresholds
    rp_df = pd.DataFrame({'Return_Period': return_periods,
                           'Q_GEV (m³/s)': q_gev.round(1),
                           'CI_2.5': ci_low_gev.round(1),
                           'CI_97.5': ci_high_gev.round(1),
                           'Q_Gumbel (m³/s)': q_gum.round(1)})
    rp_df.to_excel(writer, sheet_name='Return_Periods', index=False)

    # Sheet 4: EWS test evaluation
    test_df.to_excel(writer, sheet_name='EWS_Test_Evaluation', index=False)

    # Sheet 5: 10-year forecast
    forecast_df.to_excel(writer, sheet_name='Forecast_Monthly', index=False)
    annual_fc.to_excel(writer, sheet_name='Forecast_Annual', index=False)

    # Sheet 6: SHAP importance
    shap_df = pd.DataFrame({'Feature': FEAT_COLS_FULL, 'SHAP_Mean_Abs': shap_mean.values})
    shap_df.to_excel(writer, sheet_name='SHAP_Importance', index=False)

    # Sheet 7: RF importance
    rf_importance.reset_index().rename(columns={'index':'Feature',0:'MDI_Importance'}).to_excel(
        writer, sheet_name='RF_Importance', index=False)

    # Sheet 8: Annual statistics recap
    ann_max_q.to_excel(writer, sheet_name='Annual_Stats', index=False)

print(f'✅ Results exported → {out_xlsx}')
print('   Sheets: Model_Metrics_Test | MultiHorizon_NSE/KGE | Return_Periods |')
print('           EWS_Test_Evaluation | Forecast_Monthly/Annual | SHAP | RF | Annual_Stats')

## Cell 22 — Summary Report

In [ ]:
best_model = metrics_df['NSE'].idxmax()
print('=' * 75)
print('  SUMMARY — ML Flood Prediction | Medjerda Basin | Medjez el-Bab')
print('=' * 75)
print(f'  Period          : 1970–2024 ({len(ts_q)} monthly observations)')
print(f'  Features        : {len(FEAT_COLS_FULL)} ({FEAT_COLS_FULL})')
print(f'  Train/Val/Test  : {X_train.shape[0]} / {X_val.shape[0]} / {X_test.shape[0]} samples')
print()
print('  ── Return Period Thresholds (GEV, 95% CI) ──────────────────────────')
print(f'     Q2  = {Q2_GEV:.0f} m³/s   Q5  = {Q5_GEV:.0f} m³/s   Q10 = {Q10_GEV:.0f} m³/s')
print()
print('  ── Test Set Performance ─────────────────────────────────────────────')
print(f'  {"Model":<18} {"NSE":>7} {"KGE":>7} {"RMSE":>9} {"PBIAS":>8}')
print('  ' + '-'*52)
for model_name, row in metrics_df.iterrows():
    flag = ' ← BEST' if model_name == best_model else ''
    print(f'  {model_name:<18} {row["NSE"]:>7.3f} {row["KGE"]:>7.3f} {row["RMSE"]:>9.2f} {row["PBIAS"]:>7.1f}%{flag}')
print()
print('  ── EWS Ensemble Weights ─────────────────────────────────────────────')
print(f'     LSTM={w_lstm:.3f}  RF={w_rf:.3f}  XGBoost={w_xgb:.3f}')
print()
print('  ── 10-Year Forecast Alert Summary ──────────────────────────────────')
for level in ['VERT','JAUNE','ORANGE','ROUGE']:
    cnt = (forecast_df['Alert_Level'] == level).sum()
    print(f'     {level:<8} : {cnt:3d} months ({cnt/len(forecast_df)*100:.0f}%)')
print()
print('  ── Figures generated ────────────────────────────────────────────────')
figs = ['Fig1_EDA_TimeSeries.png','Fig2_TrendAnalysis.png','Fig3_ReturnPeriods.png',
        'Fig4_FeatureCorrelation.png','Fig5_ModelComparison.png','Fig6_Hydrographs.png',
        'Fig7_SHAP.png','Fig8_MultiHorizon.png','Fig9_LearningCurves.png',
        'Fig10_EWS.png','Fig11_10YearForecast.png']
for f in figs:
    print(f'     {f}')
print('=' * 75)